In [55]:
%load_ext autoreload
%autoreload 2

from Utils import Notebook
from Utils import Tex
from IPython.display import Math, display
import numpy as np
import scipy.linalg as la

from IPython.display import display, Math, Latex,Markdown

import ControllerDesigner


Notebook.setup()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


LaTeX has been enabled for text rendering.


### Definição da Planta

In [56]:
def check_system_properties(A, B, C, tol=1e-9):
  n = A.shape[0]
  controllability = B.copy()
  for i in range(1, n):
    controllability = np.hstack(
        (controllability, np.linalg.matrix_power(A, i) @ B))
  rank_ctrb = np.linalg.matrix_rank(controllability, tol=tol)
  controllable = rank_ctrb == n

  observability = C.copy()
  for i in range(1, n):
    observability = np.vstack(
        (observability, C @ np.linalg.matrix_power(A, i)))
  rank_obsv = np.linalg.matrix_rank(observability, tol=tol)
  observable = rank_obsv == n

  eig_A = np.linalg.eigvals(A)
  stabilizable = True

  for eig in eig_A:
    if np.real(eig) >= -tol:
      pbh_matrix = np.hstack((eig * np.eye(n) - A, B))
      if np.linalg.matrix_rank(pbh_matrix, tol=tol) < n:
        stabilizable = False
        break

  detectable = True
  for eig in eig_A:
    if np.real(eig) >= -tol:
      pbh_matrix = np.vstack((eig * np.eye(n) - A, C))
      if np.linalg.matrix_rank(pbh_matrix, tol=tol) < n:
        detectable = False
        break

  return {
      "controllability_rank": rank_ctrb,
      "observability_rank": rank_obsv,
      "controllable": controllable,
      "observable": observable,
      "stabilizable": stabilizable,
      "detectable": detectable,
      "eigenvalues": eig_A,
  }


A = np.array([
    [0.0, 1.0],
    [-4.0, 0.4]
])

B = np.array([
    [0.0],
    [1.0]
])

C = np.array([
    [1.0, 0.0]
])

results = check_system_properties(A, B, C)

print("=" * 60)
print("SYSTEM PROPERTIES")
print("=" * 60)

print(f"Controllability rank : "
      f"{results['controllability_rank']} / {A.shape[0]}")

print(f"Observability rank   : "
      f"{results['observability_rank']} / {A.shape[0]}")

print(f"Controllable         : {results['controllable']}")
print(f"Observable           : {results['observable']}")
print(f"Stabilizable         : {results['stabilizable']}")
print(f"Detectable           : {results['detectable']}")

print("\nEigenvalues of A:")
for eig in results["eigenvalues"]:
  print(f"  {eig}")

SYSTEM PROPERTIES
Controllability rank : 2 / 2
Observability rank   : 2 / 2
Controllable         : True
Observable           : True
Stabilizable         : True
Detectable           : True

Eigenvalues of A:
  (0.20000000000000012+1.98997487421324j)
  (0.20000000000000012-1.98997487421324j)


### Co-projeto do Controlador baseado em Eventos

In [57]:
import numpy as np
from IPython.display import Math, display

# -------------------------------------------------------------------------
# 1. Parâmetros do Sistema e Projeto
# -------------------------------------------------------------------------
h = 0.05
lambd = 1e-2

upsilon1 = 0.1
upsilon2 = 0.1
upsilon3 = 0.5
upsilon4 = 0.5

# Matrizes da planta
A = np.array([
    [0.0, 1.0],
    [3.75, 0.0]
], dtype=np.float64)

B = np.array([
    [0.0],
    [0.25]
], dtype=np.float64)

C = np.array([
    [0.1, 0.5]
], dtype=np.float64)

# Base ortogonal complementar para T = [C; T2]
T2 = np.array([
    [0.0, 1.0]
], dtype=np.float64)

ctrl_params = {'A': A, 'B': B, 'C': C, 'T2': T2, 'h': h, 'λ': lambd,
               'υ1': upsilon1, 'υ2': upsilon2, 'υ3': upsilon3, 'υ4': upsilon4, }

synth_res = ControllerDesigner.synthesize_output_based_setm(
    ctrl_params, eps=1e-6, verbose=False)

if synth_res is None:
  print("ERRO: Síntese Infeasible ou falha na recuperação das matrizes.")
else:
  # Ganho e Lyapunov
  K = synth_res['controller']['K']
  P = synth_res['functional']['P']
  R = synth_res['functional']['R']

  # ETM Sensor-Controlador (SC)
  Psi_sc = synth_res['etm']['sc']['Psi']
  Xi_sc = synth_res['etm']['sc']['Xi']

  # ETM Controlador-Atuador (CA)
  Psi_ca = synth_res['etm']['ca']['Psi']
  Xi_ca = synth_res['etm']['ca']['Xi']

  # Verificação rápida de estabilidade nominal contínua
  eig_cl = np.linalg.eigvals(A + B @ K)
  print("=== SÍNTESE CONCLUÍDA COM SUCESSO ===")
  print(f"Status do Solver: {synth_res['solver_status']}")
  print(f"Autovalores Nominais de Malha Fechada: {np.round(eig_cl, 4)}")

  display(Math(rf"""
    \begin{{aligned}}
        K &= {Tex.mat2tex(K)}, \qquad 
        P = {Tex.mat2tex(P)}, \qquad 
        R = {Tex.mat2tex(R)} \\[10pt]
        \Psi_{{\mathrm{{sc}}}} &= {Tex.mat2tex(Psi_sc)}, \qquad 
        \Xi_{{\mathrm{{sc}}}} = {Tex.mat2tex(Xi_sc)} \\[10pt]
        \Psi_{{\mathrm{{ca}}}} &= {Tex.mat2tex(Psi_ca)}, \qquad 
        \Xi_{{\mathrm{{ca}}}} = {Tex.mat2tex(Xi_ca)}
    \end{{aligned}}
    """))

=== SÍNTESE CONCLUÍDA COM SUCESSO ===
Status do Solver: optimal
Autovalores Nominais de Malha Fechada: [-0.6454 -1.7338]


<IPython.core.display.Math object>

### Projeto do Observador Impulsivo

In [58]:
from scipy.linalg import expm

obsr_params_discrete = {
    **ctrl_params,
    "lambda_obs": 1e0,  # Taxa mínima de decaimento por modo
    "nu_bar": 0.7       # nu_max em segundos
}

res_discrete = ControllerDesigner.synthesize_impulsive_observer(
    obsr_params_discrete,
    eps=1e-6,
    verbose=False
)

if res_discrete is not None:
  L_disc = res_discrete["L"]
  h_step = obsr_params_discrete["h"]
  n_modes = res_discrete["nu_bar_steps"]

  display(Markdown(
      rf"**Síntese Discreta Exata:** Factível ($\gamma_L = {res_discrete['gamma_L']:.4e}$)"))
  display(Markdown(
      f"* Modos testados: $\\ell = 1, \\dots, {n_modes}$ ($\\tau \\in [{h_step:.2f}, {n_modes*h_step:.2f}]$ s)"))
  print("Ganho L reconstruído:\n", np.round(L_disc, 4))

  print("\nVerificação de estabilidade modal (Raio Espectral < 1):")
  for l in range(1, n_modes + 1):
    tau_l = l * h_step
    Phi_l = expm(ctrl_params["A"] * tau_l)
    A_err_mode = (
        np.eye(ctrl_params["A"].shape[0]) - L_disc @ ctrl_params["C"]) @ Phi_l
    rho_mode = np.max(np.abs(np.linalg.eigvals(A_err_mode)))
    print(
        f"Modo l={l:2d} (tau={tau_l:.2f}s) -> Raio Espectral = {rho_mode:.4f}")
else:
  display(Markdown(
      "**Síntese Discreta:** Infactível para a taxa de decaimento informada."))

**Síntese Discreta Exata:** Factível ($\gamma_L = 1.6652e-05$)

* Modos testados: $\ell = 1, \dots, 14$ ($\tau \in [0.05, 0.70]$ s)

Ganho L reconstruído:
 [[0.925 ]
 [1.7913]]

Verificação de estabilidade modal (Raio Espectral < 1):
Modo l= 1 (tau=0.05s) -> Raio Espectral = 0.9077
Modo l= 2 (tau=0.10s) -> Raio Espectral = 0.8239
Modo l= 3 (tau=0.15s) -> Raio Espectral = 0.7479
Modo l= 4 (tau=0.20s) -> Raio Espectral = 0.6789
Modo l= 5 (tau=0.25s) -> Raio Espectral = 0.6162
Modo l= 6 (tau=0.30s) -> Raio Espectral = 0.5594
Modo l= 7 (tau=0.35s) -> Raio Espectral = 0.5077
Modo l= 8 (tau=0.40s) -> Raio Espectral = 0.4609
Modo l= 9 (tau=0.45s) -> Raio Espectral = 0.4184
Modo l=10 (tau=0.50s) -> Raio Espectral = 0.3797
Modo l=11 (tau=0.55s) -> Raio Espectral = 0.3447
Modo l=12 (tau=0.60s) -> Raio Espectral = 0.3129
Modo l=13 (tau=0.65s) -> Raio Espectral = 0.2840
Modo l=14 (tau=0.70s) -> Raio Espectral = 0.2578


In [59]:
import numpy as np


def format_matrix_to_cpp(mat, name="X", scientific=True):
  """Converte um vetor ou matriz numpy para o formato C++ com chave única:

  X = {val1, val2, val3, ...};
  """
  mat = np.atleast_2d(mat)
  flat_vals = mat.flatten()

  fmt = "{:.2e}" if scientific else "{:.4f}"
  vals_str = ", ".join([fmt.format(val) for val in flat_vals])

  return f"{name} = {{{vals_str}}};"


# Extração das variáveis do novo escopo Dual-Channel
Xi_sc = synth_res["etm"]["sc"]["Xi"]
Psi_sc = synth_res["etm"]["sc"]["Psi"]
Xi_ca = synth_res["etm"]["ca"]["Xi"]
Psi_ca = synth_res["etm"]["ca"]["Psi"]
K = synth_res["controller"]["K"]
L = res_discrete["L"]

# Exibição no console / Jupyter no formato exato solicitado
print(format_matrix_to_cpp(Xi_sc, name="Xi_sc"))
print(format_matrix_to_cpp(Psi_sc, name="Psi_sc"))
print(format_matrix_to_cpp(Xi_ca, name="Xi_ca"))
print(format_matrix_to_cpp(Psi_ca, name="Psi_ca"))
print(format_matrix_to_cpp(K, name="K"))
print(format_matrix_to_cpp(L, name="L"))

Xi_sc = {6.44e+00};
Psi_sc = {1.02e+00};
Xi_ca = {2.17e+03, -9.79e+02, -9.79e+02, 4.42e+02};
Psi_ca = {6.59e-01, 1.33e+00, 1.33e+00, 2.94e+00};
K = {-1.95e+01, -9.52e+00};
L = {9.25e-01, 1.79e+00};
